# 07 — Load only the requested expression table

Run this inspection notebook in the SpatialData environment. Worker launch notebooks themselves do not require TensorFlow, scVI or RAPIDS imports.
Selected expression is loaded into RAM. Images remain lazy; shapes and metadata may still be large.
This notebook reads the local active product; published S3 copies must be staged explicitly under the same approved policy.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Choose a built sample

In [ ]:
row = next(r for r in PROJECT["rows"] if r["goal"] != "export_only")
p = sample_layout(PROJECT, row)
print(p["store"])
from vhd.io import load_table, load_spatial_with_table, table_metadata
# Change to "bins_2um" for bin-only output.
TABLE = "cells" if row["goal"] != "bin_QCed" else "bins_2um"
obs, var = table_metadata(p["store"], TABLE)
print(obs.shape, var.shape)
display(obs.head())

## Optional matrix load (explicit RAM use)

In [ ]:
LOAD_EXPRESSION = False
if LOAD_EXPRESSION:
    selected = load_table(p["store"], TABLE)
    print(selected)
# After notebook 11: load_table(p["store"], "cells", merge_scvi=True)